In [2]:
import numpy as np
import os, json
import pandas as pd
import librosa
from collections import defaultdict
import statistics

### Descriptive analysis

(1) How many stories? Average production duration

(2) How many gold-standard sentences on average per stories, average sentence length

(3) How many actually produced sentences by children, on average per stories, average produced sentence length

(4) How many parent sentences, repetitions, run-ons, etc

(5) Error distributions

(6) Word correct per minute

In [61]:
sentenceLabels_data = pd.read_csv('/Users/liu.ying/Spaceship/Storiza_speech_modeling/processed_data/sentenceLabels_with_comments.csv')

In [16]:
temp_annotated_intended_sentences_list = sentenceLabels_data['goldStandard'].tolist()
temp_original_audio_list = sentenceLabels_data['original_audio_name'].tolist()
temp_sentenceLabels_start_time_list = sentenceLabels_data['start_time'].tolist()
temp_sentenceLabels_end_time_list = sentenceLabels_data['end_time'].tolist()
repeated_list = sentenceLabels_data['repeated'].tolist()
runon_list = sentenceLabels_data['runon'].tolist()
nonchild_list = sentenceLabels_data['nonchild'].tolist()

draft_ordered_sentences_list = sentenceLabels_data['ordered_sentences'].tolist()
temp_ordered_sentences_list = []

annotated_intended_sentences_list = []
original_audio_list = []
sentenceLabels_start_time_list = []
sentenceLabels_end_time_list = []

for i in range(len(nonchild_list)):
    nonchild = nonchild_list[i]
    if not nonchild:
        annotated_intended_sentences_list.append(temp_annotated_intended_sentences_list[i])
        original_audio_list.append(temp_original_audio_list[i])
        sentenceLabels_start_time_list.append(temp_sentenceLabels_start_time_list[i])
        sentenceLabels_end_time_list.append(temp_sentenceLabels_end_time_list[i])
        temp_ordered_sentences_list.append(draft_ordered_sentences_list[i])

In [17]:
num_repeated = 0
num_runon = 0
num_nonchild = 0
for i in range(len(repeated_list)):
    repeated = repeated_list[i]
    runon = runon_list[i]
    nonchild = nonchild_list[i]
    if repeated:
        num_repeated += 1
    if runon:
        num_runon += 1
    if nonchild:
        num_nonchild += 1

print('Number of repeated sentences is', num_repeated)
print('Number of run-on sentences is', num_runon)
print('Number of non-child sentences is', num_nonchild)

Number of repeated sentences is 24
Number of run-on sentences is 208
Number of non-child sentences is 6


In [29]:
unique_original_audios = list(set(original_audio_list))
unique_original_audios = [audio for audio in unique_original_audios if not pd.isna(audio)]
num_stories = len(unique_original_audios)
print('Number of produced stories is', num_stories)
print('Number of produced utterances is', len(annotated_intended_sentences_list))

Number of produced stories is 321
Number of produced utterances is 3025


In [ ]:
import ast

list

In [81]:
utterance_duration_list = []
story_duration_dict = defaultdict(list)

ordered_sentences_list = [] # list of intended sentences from the original story (not the annotated intended sentences)

ordered_sentences_dict = {}
for audio in unique_original_audios:
    if audio not in story_duration_dict:
        story_duration_dict[audio] = []
        ordered_sentences_dict[audio] = []
        for i in range(len(sentenceLabels_start_time_list)):
            ordered_sentences = temp_ordered_sentences_list[i]
        #    try:
            ordered_sentences = ast.literal_eval(ordered_sentences)
            if ordered_sentences not in ordered_sentences_list:
                ordered_sentences_list.append(ordered_sentences)
                
        #    except:
        #        pass
        
            start_time = float(sentenceLabels_start_time_list[i])
            end_time = float(sentenceLabels_end_time_list[i])
            duration = end_time - start_time
            if audio == 'uid_TzE1wYChYkTUAVNeVtZjXuO8cBi2_sid_xVtZiYDJKDrpbrcttqPg_1743434705.wav':
                print(start_time, end_time, duration)
            utterance_duration_list.append(duration)
            if original_audio_list[i] == audio:
                story_duration_dict[audio].append(duration)
                ordered_sentences_dict[audio] = ordered_sentences

for k, v in ordered_sentences_dict.items():
    print(k, v)

ave_story_duration = sum(utterance_duration_list) / num_stories
if ave_story_duration < 60:
    ave_story_duration = round(ave_story_duration, 2)
    print('Average story duration is', ave_story_duration, 'seconds')
else:
    ave_story_min = round(ave_story_duration // 60)
    ave_story_secs = round(ave_story_duration % 60)
    print('Average story duration is', str(ave_story_min) + 'min' + str(ave_story_secs) + 's')


story_duration_list = []
for k, v in story_duration_dict.items():
    story_duration_list.append(sum(v))

shortest_story_duration = min(story_duration_list)
if shortest_story_duration < 60:
    shortest_story_duration = round(shortest_story_duration, 2)
    print('Shortest story duration is', shortest_story_duration, 'seconds')
else:
    shortest_story_min = round(shortest_story_duration // 60)
    shortest_story_secs = round(shortest_story_duration % 60)
    print('Shortest story duration is', str(shortest_story_min) + 'min' + str(shortest_story_secs) + 's')

longest_story_duration = max(story_duration_list)
if longest_story_duration < 60:
    longest_story_duration = round(longest_story_duration, 2)
    print('Longest story duration is', longest_story_duration, 'seconds')
else:
    longest_story_min = round(longest_story_duration // 60)
    longest_story_secs = round(longest_story_duration % 60)
    print('Longest story duration is', str(longest_story_min) + 'min' + str(longest_story_secs) + 's')

ave_utterance_duration = statistics.mean(utterance_duration_list)
if ave_utterance_duration < 60:
    ave_utterance_duration = round(ave_utterance_duration, 2)
    print('Average utterance duration is', ave_utterance_duration, 'seconds')
else:
    ave_utterance_min = round(ave_utterance_duration // 60)
    ave_utterance_secs = round(ave_utterance_duration % 60)
    print('Average utterance duration is', str(ave_utterance_min) + 'min' + str(ave_utterance_secs) + 's')

shortest_utterance_duration = min(utterance_duration_list)
if shortest_utterance_duration < 60:
    shortest_utterance_duration = round(shortest_utterance_duration, 2)
    print('Shortest utterance duration is', shortest_utterance_duration, 'seconds')
else:
    shortest_utterance_min = round(shortest_utterance_duration // 60)
    shortest_utterance_secs = round(shortest_utterance_duration % 60)
    print('Shortest utterance duration is', str(shortest_utterance_min) + 'min' + str(shortest_utterance_secs) + 's')

longest_utterance_duration = max(utterance_duration_list)
if longest_utterance_duration < 60:
    longest_utterance_duration = round(longest_utterance_duration, 2)
    print('Longest utterance duration is', longest_utterance_duration, 'seconds')
else:
    longest_utterance_min = round(longest_utterance_duration // 60)
    longest_utterance_secs = round(longest_utterance_duration % 60)
    print('Longest utterance duration is', str(longest_utterance_min) + 'min' + str(longest_utterance_secs) + 's')

#for k, v in ordered_sentences_dict.items():
#    print(k, v)


3.1734557118402047 18.345426558896467 15.171970847056262
22.41030241451201 39.2403147289904 16.830012314478388
41.41118031428928 55.682110145675125 14.270929831385843
57.02277979598388 79.07180958205363 22.049029786069752
79.39866705878178 99.07659514926412 19.67792809048234
99.53641159957664 110.16 10.623588400423358
0.1136135754242066 2.006250537010425 1.8926369615862184
2.0201360979287384 8.748245307663913 6.728109209735175
8.760306278952708 16.325101990487262 7.564795711534554
16.6153531337708 25.06326925306517 8.44791611929437
25.344320930309053 29.49477126703516 4.150450336726106
29.902749508195637 36.367128036160814 6.464378527965177
37.99482083346767 47.34569551999613 9.350874686528456
47.93529007345238 53.85283475107396 5.917544677621578
53.97551450963506 61.05935398523162 7.083839475596555
61.07199832448545 71.4568781172877 10.384879792802252
71.47315026981474 87.97435762415346 16.501207354338717
88.44351397095882 93.0310026382299 4.587488667271074
94.92483423853756 96.580605

In [39]:
num_ordered_sentences = 0
for sent_list in ordered_sentences_list:
    num_ordered_sentences += len(sent_list)

print('Number of ordered intended sentences from the original stories is', num_ordered_sentences)

Number of ordered intended sentences from the original stories is 3121


In [55]:
# Grouping annotated intended sentences by original audio names
audio_annotated_intended_sentences = defaultdict(list)

for audio in unique_original_audios:
    audio_annotated_intended_sentences[audio] = []
    for i in range(len(annotated_intended_sentences_list)):
        if original_audio_list[i] == audio:
            sentence = annotated_intended_sentences_list[i]
            audio_annotated_intended_sentences[audio].append(sentence)


In [56]:
for audio in unique_original_audios:
    annotated_intended_sentences = audio_annotated_intended_sentences[audio]
    original_intended_sentences = ordered_sentences_dict[audio]
    if len(annotated_intended_sentences) != len(original_intended_sentences):
        print(annotated_intended_sentences)
        print(original_intended_sentences)
        print('\n')

['Once, a dragon prince named Cole lived in a stone castle.']
['Once, a dragon prince named Cole lived in a stone castle.', ' Cole loved to ride his noble steed, Hope. ', 'One day, Cole and Hope found a lone dove trapped in rope. ', 'Cole came close and freed the dove. ', 'Grateful, the dove promised to help Cole in the future. ', 'Later, Cole needed to cross a vast ocean, but no boat was near. ', 'The dove returned with a large leaf to float on. ', 'Cole made it home safe, feeling happy and grateful. ', 'The dragon prince knew that helping others would always bring hope and joy.']


['In a magical land, a unicorn named Stell galloped through a tall hall.', 'Stell was the best detective and could solve any call.', 'One day, Stell got a call about a lost doll.']
['In a magical land, a unicorn named Stell galloped through a tall hall.', 'Stell was the best detective and could solve any call.', 'One day, Stell got a call about a lost doll.', ' The doll was said to be small and full of mag

In [ ]:
annotated_intended_sent_len_list = []
for k, v in audio_annotated_intended_sentences.items():
    for sent in v:
        sent_len = len(sent.split())
        annotated_intended_sent_len_list.append(sent_len)
        
print('Number of annotated intended sentences is', len(annotated_intended_sent_len_list))
print('Average sentence length for annotated intended sentences is', round(statistics.mean(annotated_intended_sent_len_list)))

Number of annotated intended sentences is 3025
Average sentence length for annotated intended sentences is 10 2


In [ ]:
original_intended_sent_len_list = []
for sent_list in ordered_sentences_list:
    for sent in sent_list:
        sent_len = len(sent.split())
        original_intended_sent_len_list.append(sent_len)

print('Number of original intended sentences is', len(original_intended_sent_len_list))
print('Average sentence length for original intended sentences is', round(statistics.mean(original_intended_sent_len_list)))

Number of original intended sentences is 3121
Average sentence length for original intended sentences is 10 2


In [80]:
## Word correct per minute, based on the corpus data
word_segments_data = pd.read_csv('/Users/liu.ying/Spaceship/Storiza_speech_modeling/processed_data/word_level_data.csv')
word_path_list = word_segments_data['Path'].tolist()
word_error_categories_list = word_segments_data['Error Category'].tolist()
word_error_labels_list = word_segments_data['Error Labels'].tolist()

story_error_categories_dict = {}
story_error_labels_dict = {}
for audio in unique_original_audios:
    story_error_categories_dict[audio] = []
    story_error_labels_dict[audio] = []
    audio_name = audio.split('.')[0]
    for i in range(len(word_path_list)):
        path = word_path_list[i].split('/')[-1]    
        if path.startswith(audio_name):
            error_categories = word_error_categories_list[i].split('+')
            if error_categories[0] == "Mixed Error":
                error_categories = error_categories[1 : ]
            for category in error_categories:
                story_error_categories_dict[audio].append(category)
            error_labels = ast.literal_eval(word_error_labels_list[i])
            for label in error_labels:
                story_error_categories_dict[audio].append(label)
        else:
            pass

word_correct_per_min_list = []
for k, v in story_error_categories_dict.items():
    if v != []:
        print(v)
        print(story_duration_dict[k])
        story_duration = sum(story_duration_dict[k])
        num_word_correct = v.count('Correct')
        print(story_duration, num_word_correct)
        word_correct_per_min = (num_word_correct / story_duration) * 60
        word_correct_per_min_list.append(word_correct_per_min)

statistics.mean(word_correct_per_min_list)


['Disfluency Error', 'Repair', 'Broken Word', 'Correct', 'Disfluency Error', 'Parental Aid', 'Correct', 'Correct', 'Orthographic Sub.', 'Phonological', 'Disfluency Error', 'Phonological', 'Vowel Substitution', 'Prolongation']
[15.171970847056262, 16.830012314478388, 14.270929831385843, 22.049029786069752, 19.67792809048234, 10.623588400423358]
98.62345926989595 3


1.825123569306229

In [ ]:
## 
word_segments_data_file = '/Users/liu.ying/Spaceship/Storiza_speech_modeling/processed_data/word_level_data.csv'
full_word_segments_data = pd.read_csv(word_segments_data_file)

word_segments_data = full_word_segments_data #[:8] ## Taking out a sub-sample to make sure code runs

In [24]:
word_segments_data["Error Category"] = word_segments_data["Error Category"].apply(lambda x: [category.strip() for category in x.split("+") if category != 'Mixed Error'])

In [15]:
word_segments_data['Error Labels'].tolist()

["['Consonant Substitution', 'Phonological']",
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]',
 '[]']

In [21]:
for tok in word_segments_data['Error Labels'].tolist():
    print(tok.strip('[]').split(', '))

["'Consonant Substitution'", "'Phonological'"]
['']
['']
['']
['']
['']
['']
['']


In [22]:
word_segments_data['Error Labels'] = word_segments_data['Error Labels'].apply(lambda labels: labels.strip('[]').split(', ') if labels != '[]'  else ['NONE'])

/var/folders/kv/k9vdbjw913xbm12v5wfkt_zw0000gq/T/ipykernel_13489/2337123228.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word_segments_data['Error Labels'] = word_segments_data['Error Labels'].apply(lambda labels: labels.strip('[]').split(', ') if labels != '[]'  else ['NONE'])


In [25]:
all_error_categories = sorted({category for sublist in word_segments_data["Error Category"] for category in sublist})
num_error_categories = len(all_error_categories)
print('Error Categories:', all_error_categories)
print('Total number of unique error categories:', len(all_error_categories))

Error Categories: ['Contraction/Shortening', 'Correct', 'Disfluency Error', 'Grammatical', 'Grammatical Error', 'Orthographic Error', 'Orthographic Sub.', 'Other', 'Phonological', 'Phonological Error', 'Run-on', 'Run-on Word', 'Self Response', 'Structural', 'Structural Error', 'Unintelligible', 'Visual Tracking', 'Visual Tracking Error', 'Whispering']
Total number of unique error categories: 19


In [26]:
category2id = {category : i for i, category in enumerate(all_error_categories)}
id2category = {i : category for i, category in enumerate(all_error_categories)}

In [27]:
# Convert to multi-hot vectors
def encode_labels(error_category_list):
    vec = [0] * num_error_categories
    for category in error_category_list:
        vec[category2id[category]] = 1
    return vec

In [28]:
word_segments_data['labels'] = word_segments_data['Error Category'].apply(encode_labels)

In [ ]:
word_segments_data['labels'].tolist()[:2] # A list of multi-hot vectors

[[0, 1, 1], [1, 0, 0]]